# Quantize HF Model (Drive Pipeline)

This notebook runs in Colab and stores both full + quantized HF model folders in Google Drive.


In [6]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

candidates = [
    Path('/content/drive/MyDrive/training-embedding'),
    Path('/content/drive/My Drive/training-embedding'),
]

project_root = next((p for p in candidates if p.exists()), None)
if project_root is None:
    raise RuntimeError('training-embedding folder not found under mounted Drive.')

print(f'project_root={project_root}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
project_root=/content/drive/MyDrive/training-embedding


In [7]:
import os

def _read_dotenv_value(dotenv_path: Path, key: str) -> str:
    if not dotenv_path.exists():
        return ''
    for raw_line in dotenv_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        k = k.strip()
        if k.startswith('export '):
            k = k[len('export '):].strip()
        if k == key:
            return v.strip().strip('\"').strip("'")
    return ''

hf_token = os.environ.get('HF_TOKEN', '').strip()
if not hf_token:
    hf_token = _read_dotenv_value(project_root / '.env', 'HF_TOKEN')

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN configured for this runtime.')
else:
    print('HF_TOKEN not found. Download will proceed unauthenticated.')


HF_TOKEN configured for this runtime.


In [8]:
MODEL_ID = 'ytu-ce-cosmos/Turkish-Gemma-9b-T1'
DRIVE_MODELS_ROOT = project_root / 'models'
FULL_MODEL_DIR = '/content/models/turkish-gemma-9b-t1-full'
QUANT_OUTPUT_DIR = DRIVE_MODELS_ROOT / 'turkish-gemma-9b-t1-4bit'
BITS = 4
TORCH_DTYPE = 'float16'

DRIVE_MODELS_ROOT.mkdir(parents=True, exist_ok=True)

print('MODEL_ID:', MODEL_ID)
print('FULL_MODEL_DIR:', FULL_MODEL_DIR)
print('QUANT_OUTPUT_DIR:', QUANT_OUTPUT_DIR)
print('BITS:', BITS, 'TORCH_DTYPE:', TORCH_DTYPE)


MODEL_ID: ytu-ce-cosmos/Turkish-Gemma-9b-T1
FULL_MODEL_DIR: /content/models/turkish-gemma-9b-t1-full
QUANT_OUTPUT_DIR: /content/drive/MyDrive/training-embedding/models/turkish-gemma-9b-t1-4bit
BITS: 4 TORCH_DTYPE: float16


In [9]:
%%bash
set -euo pipefail
python -m pip install -U pip
python -m pip install -U "bitsandbytes>=0.46.1" accelerate transformers peft huggingface_hub


In [12]:
import os, subprocess, sys

cmd = [
    sys.executable,
    str(project_root / 'hf_quantize_model.py'),
    '--model-id', MODEL_ID,
    '--download-dir', str(FULL_MODEL_DIR),
    '--output-dir', str(QUANT_OUTPUT_DIR),
    '--bits', str(BITS),
    '--torch-dtype', TORCH_DTYPE,
]

proc = subprocess.run(cmd, env=os.environ.copy(), text=True, capture_output=True)
print("returncode:", proc.returncode)
print("----- STDOUT -----")
print(proc.stdout[-8000:])
print("----- STDERR -----")
print(proc.stderr[-8000:])


returncode: 0
----- STDOUT -----
Saved 4-bit quantized model to: /content/drive/MyDrive/training-embedding/models/turkish-gemma-9b-t1-4bit

----- STDERR -----
, Materializing param=model.layers.39.pre_feedforward_layernorm.weight] 
Loading weights: 100%|██████████| 464/464 [01:16<00:00,  6.08it/s, Materializing param=model.norm.weight]

Writing model shards: 100%|██████████| 1/1 [03:40<00:00, 220.13s/it]



In [ ]:
# verify quantized model is installed
def _size_gib(path: Path) -> float:
    if path.is_file():
        return path.stat().st_size / (1024**3)
    total = 0
    for p in path.rglob('*'):
        if p.is_file():
            total += p.stat().st_size
    return total / (1024**3)


if QUANT_OUTPUT_DIR:
    print(f'Quantized model size: {_size_gib(QUANT_OUTPUT_DIR):.2f} GiB')

print('\nQuantized folder contents:')
for p in sorted(QUANT_OUTPUT_DIR.glob('*')):
    suffix = '/' if p.is_dir() else ''
    print('-', p.name + suffix)


Quantized model size: 5.74 GiB

Quantized folder contents:
- chat_template.jinja
- config.json
- generation_config.json
- model.safetensors
- tokenizer.json
- tokenizer_config.json


In [ ]:
# Optional cleanup after successful quantization
# This removes the full-precision model directory from Drive to save space.
REMOVE_FULL_MODEL_FROM_DRIVE = True

import shutil

if REMOVE_FULL_MODEL_FROM_DRIVE:
    shutil.rmtree(FULL_MODEL_DIR, ignore_errors=True)
    print(f'Removed: {FULL_MODEL_DIR}')
else:
    print('Cleanup skipped.')
